# Making Roman image cutouts with Gladius

This notebook opens Gladius in your browser. You can choose images on your
computer or find images in the MAST archive, add a list of targets, and make
cutouts around them. The form explains the settings and shows any errors.

The interface needs no MAST token when you use local images. Archive access
uses a token for your MAST account. A separate example later in this notebook
shows how to query the archive and review cutouts from Python.


In [ ]:
from pathlib import Path

WORKSPACE = Path.cwd() / "gladius-archive-review"
RUN_GUI = True


## Opening Gladius

The next cell opens the interface inside this notebook and prints a link for
opening it in a separate tab. Gladius reads and writes files on the computer
running the notebook. `WORKSPACE` above chooses its working folder.

On Nexus, your Jupyter login lets the browser reach Gladius through the server’s
proxy. Image paths refer to files on Nexus.


In [ ]:
if RUN_GUI:
    import threading
    from IPython.display import IFrame, display
    from gladius.web import GuiServer

    if "gui_thread" not in globals() or not gui_thread.is_alive():
        gui_server = GuiServer(WORKSPACE, port=0, nexus=True, iframe_origin="self", initial_config=globals().get("config"))
        gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
        gui_thread.start()
    gladius_ui_url = gui_server.public_url
    print("Open the GUI in a separate tab:", gladius_ui_url)
    display(IFrame(src=gladius_ui_url, width="100%", height=650))
else:
    print("GUI skipped. Set RUN_GUI=True in the settings cell to explore it.")


## Stopping Gladius

Gladius keeps running when you close its frame or browser tab. Once your job
finishes, you can stop it by changing `STOP_GUI` to `True` and running the next
cell. Leaving it `False` keeps the interface open when you run the whole notebook.


In [ ]:
STOP_GUI = False  # True stops the interface after any active job finishes.

if STOP_GUI:
    if "gui_thread" in globals() and gui_thread.is_alive():
        gui_server.shutdown()
        gui_server.server_close()
        gui_thread.join(timeout=5)
        print("Embedded GUI stopped.")


## Working from Python instead

The rest of this notebook shows how to select one MAST image, make cutouts,
and inspect the results from Python. You can skip this example if you are using
the interface above.

The example downloads a full detector image. It checks the size that MAST reports
before downloading and keeps the image so you can compare it with the cutouts.
Images, coordinates, and notebook outputs from restricted observations need to
stay private.


### Choosing an image and credentials

The settings below describe the archive example. `RUN_ARCHIVE_EXAMPLE = True`
enables its code cells; the default leaves them off.

`ENV_FILE` is the path to a file containing your MAST token as
`MAST_API_TOKEN=...`. The token belongs in that file, rather than in this notebook.
You can create one on the [MAST token page](https://auth.mast.stsci.edu/tokens).
Your account must have permission to read the image.

`DATASET_ID` identifies one Roman observation. MAST calls this field
`fileSetName`; you can find it in the observation’s archive metadata.
`MAX_PARENT_MIB` sets the largest advertised image size you want to download.

`TARGET_CSV` can point to your target list, with `id,ra,dec` in ICRS degrees or
`id,x,y` in zero-based image pixels. Without a target list, this example uses the
archive’s image centre as a practice location. That location does not identify
a particular star.


In [ ]:
from pathlib import Path

RUN_ARCHIVE_EXAMPLE = False
ENV_FILE = None
# ENV_FILE = Path("/path/to/.env")
DATASET_ID = None
# DATASET_ID = "your-exact-Roman-fileSetName"
TARGET_CSV = None
# TARGET_CSV = Path("/path/to/targets.csv")
CUTOUT_SIZE = 150
MAX_PARENT_MIB = 512


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    import os
    from dotenv import dotenv_values

    if ENV_FILE is None:
        raise ValueError("Set ENV_FILE to the path of your local credential file, then rerun this cell.")
    ENV_FILE = Path(ENV_FILE).expanduser().resolve()
    if not ENV_FILE.is_file():
        raise FileNotFoundError("ENV_FILE must point to an existing file; no archive request was made.")
    credentials = dotenv_values(ENV_FILE, interpolate=False)
    if not (credentials.get("MAST_API_TOKEN") or "").strip():
        raise ValueError("The selected file must contain a nonempty MAST_API_TOKEN.")
    os.environ["MAST_API_TOKEN"] = credentials["MAST_API_TOKEN"]
    del credentials

    WORKSPACE = Path(WORKSPACE).expanduser().resolve()
    if not isinstance(CUTOUT_SIZE, int) or isinstance(CUTOUT_SIZE, bool) or CUTOUT_SIZE < 1:
        raise ValueError("CUTOUT_SIZE must be a positive integer.")
    if not isinstance(MAX_PARENT_MIB, (int, float)) or MAX_PARENT_MIB <= 0:
        raise ValueError("MAX_PARENT_MIB must be positive.")
    print("Credential file loaded. Token is not displayed.")


### Checking the request

The next cell prepares a request for the exact dataset and Level 2 products.
It prints the request without contacting MAST. The response limit checks for
unexpected matches; the exact dataset identifier is what keeps the search narrow.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    if not isinstance(DATASET_ID, str) or not DATASET_ID.strip():
        raise ValueError("Set DATASET_ID to one exact Roman fileSetName. No MAST query was sent.")
    DATASET_ID = DATASET_ID.strip()
    if not all(c.isalnum() or c in "_-" for c in DATASET_ID):
        raise ValueError("Use an exact dataset ID, without wildcards, lists or ranges.")
    metadata_request = {"fileSetName": DATASET_ID, "productLevel": "2"}
    print("Metadata criteria:", metadata_request)
    print("Response limit: 2; expected exact match: 1 dataset. No download yet.")


### Finding the calibrated image

The next cell signs in to MAST and sends that request. It lists calibrated ASDF
products for the selected dataset and chooses the smallest one with a known size.
It checks that size against `MAX_PARENT_MIB`. This cell reads archive metadata;
it does not download the image.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    import json
    import math
    import numpy as np
    from astropy.table import Table
    from IPython.display import display
    from astroquery.mast import MastMissions
    from gladius import RunConfig, Target, read_targets_csv, run

    missions = MastMissions(mission="roman")
    missions.TIMEOUT = 30
    missions.login(token=os.environ["MAST_API_TOKEN"], store_token=False)
    observations = missions.query_criteria(
        **metadata_request, limit=2,
        select_cols=["fileSetName", "ra", "dec", "detector", "optical_element", "exposure_start_time"])
    if len(observations) != 1 or str(observations[0]["fileSetName"]) != DATASET_ID:
        raise ValueError("Expected exactly the requested dataset. Narrow/fix DATASET_ID; no download made.")
    products = missions.filter_products(missions.get_unique_product_list(observations), file_suffix="_cal")
    candidates = []
    for product in products:
        name = str(product["filename"])
        try:
            size = float(product["size"])
        except (ValueError, TypeError, KeyError):
            continue
        if name.endswith("_cal.asdf") and math.isfinite(size) and size > 0 and size == int(size):
            candidates.append((int(size), name))
    if not candidates:
        raise ValueError("No calibrated 2D product with a known size in this dataset; no download made.")
    parent_bytes, product_filename = min(candidates)
    if parent_bytes > MAX_PARENT_MIB * 1024**2:
        raise ValueError(f"Advertised parent size is {parent_bytes/1024**2:.1f} MiB; exceeds MAX_PARENT_MIB.")
    display(observations)
    print("Exact product:", product_filename)
    print("Advertised download:", round(parent_bytes/1024**2, 2), "MiB")


### Describing the cutouts

`RunConfig` holds the same settings as a YAML configuration. The next cell sets
the targets, selected archive product, cutout size, and output formats. It also
asks Gladius to keep the original image for comparison.

The science array contains pixel values, the uncertainty array contains their
reported errors, and the quality array contains flags such as saturation.
Gladius copies these arrays without rotating or interpolating the pixels.
Its coordinate model, or WCS, maps pixels to positions on the sky.

The optional quality screen looks for elongated image structure near the target.
Its threshold here is an example, not a calibrated recommendation. A flagged
cutout remains available even when Gladius excludes it from display products.

The [configuration guide](https://roman-gladius.readthedocs.io/en/latest/configuration.html)
explains all settings. Creating a configuration does not contact MAST.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    if TARGET_CSV is None:
        targets = [Target(id="archive-centre-not-a-star-id",
                          ra=float(observations[0]["ra"]), dec=float(observations[0]["dec"]))]
    else:
        targets = read_targets_csv(Path(TARGET_CSV).expanduser())
    config = RunConfig(
        run={"name": "archive-field-review"},
        targets=targets,
        inputs={"mode": "mast", "mast_criteria": {"fileSetName": DATASET_ID},
                "product_filename": product_filename, "query_limit": 2, "mast_radius_arcmin": 6},
        extraction={"size": CUTOUT_SIZE, "edge": "pad"},
        astrometry={"mode": "validate", "roundtrip_tolerance_pixels": 0.01},
        quality={"autocorrelation_limit": 2.6, "tile_size": 64, "radius": 15,
                 "exclude_from_stacks": True},
        stacks={"enabled": False, "gif_enabled": False, "rgb_enabled": False},
        storage={"output_directory": "output", "retain_full_images": True,
                 "formats": ["asdf", "fits"]},
    ).resolved(WORKSPACE)


### Reviewing the configuration

The next cell prints the archive queries that Gladius will use and saves the
configuration as YAML. Each query must still contain the selected dataset.
A sky-position search finds candidate images; it does not guarantee that the
target lies on the detector. Gladius checks that when it processes the image.

For a larger sample, the selection needs appropriate program, date, detector,
and filter limits. Even a small cutout requires the full image download.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    import yaml
    from gladius.inputs import mast_queries

    queries = mast_queries(config)
    assert all(query.get("fileSetName") == DATASET_ID for query in queries)
    print("Pipeline query count:", len(queries))
    for query in queries:
        print(query)
    print("Exact downloaded product:", config.inputs.product_filename)
    print(yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    (WORKSPACE / "review-config.yaml").write_text(
        yaml.safe_dump(config.model_dump(mode="json"), sort_keys=False))


### Making cutouts

The next cell downloads the selected image and makes the cutouts. Gladius opens
the image once for all targets and keeps it for the previews below. If matching
work already exists in the cache, Gladius can reuse it. The progress messages
and manifest record what it did.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    manifest_path = run(config, progress=print)
    manifest = json.loads(manifest_path.read_text())
    print("Manifest:", manifest_path)
    print("Work performed:", manifest["stage_counts"])


### Checking the results

The table below includes completed, skipped, and failed attempts. A completed
cutout with `accepted=False` failed the optional quality screen; Gladius still
saved its pixels. A `target_off_detector` entry means the target position was
outside that image. Processing errors are separate from quality flags.

The manifest records the run’s status, selected inputs, and errors. These records
help explain missing results; output files alone do not show that a run succeeded.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    from astropy.table import Table
    from IPython.display import display
    from gladius.inspection import audit_rows, contact_sheets, parent_preview

    # A Python list of dictionaries: one row for every attempted input/target pair.
    review = audit_rows(manifest_path)
    audit = Table(rows=review)
    display(audit[["target", "parent", "filter", "status", "accepted", "display_eligible", "axis_ratio", "reason"]])
    display(audit[["target", "parent", "padding_fraction", "do_not_use_fraction", "saturated_fraction"]])
    print("Run status:", manifest["status"])
    print("Input images used:", len(manifest["sources"]))
    print("Parents rejected by date/detector/filter:", manifest["selection_rejections"])
    print("Recorded errors:", manifest["errors"])


### Comparing cutouts with the original image

The previews below draw each requested cutout boundary and target position on
the original image. A boundary can cross the detector edge when padding is on.
Purple pixels in a cutout preview mark unavailable values.

The crosses show the positions you supplied, rather than measured star centres.
The north and east arrows come from the image’s WCS. Comparing an image with its
own cutout checks placement within that image; it does not independently verify
its sky coordinates.

An independent finding chart can help you identify the intended star. The
optional cell displays your chart beside the results, but does not align images,
match stars, or certify an identification.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    # Show every retained/available parent, once, with all its completed cutout boxes.
    parent_ids = sorted({entry["parent_sha256"] for entry in manifest["entries"].values()
                         if entry["status"] == "complete"})
    for parent_id in parent_ids:
        display(parent_preview(manifest_path, parent_id, width=800))


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    from PIL import Image

    FINDING_CHART = None
    # FINDING_CHART = Path("/path/to/finding-chart.png")  # Optional independent chart.
    if FINDING_CHART is not None:
        with Image.open(FINDING_CHART) as chart:
            reference = chart.convert("RGB")
            reference.thumbnail((900, 900))
            display(reference)


### Reviewing every cutout

The contact sheets include all completed cutouts, including those the screen
flagged. Each cross marks a requested target position. The gold square shows
the region used by the quality screen, so defects outside it can go unnoticed.

The score measures elongation in the image’s autocorrelation: its similarity to
shifted copies of itself. Scores near one describe a rounder pattern; larger
scores describe a more elongated pattern. This is not a fitted stellar shape.
An insufficient-signal or insufficient-tile result means the screen could not
make a useful measurement, rather than that it detected trailing.

Images in the same target, filter, and unit group share a brightness stretch.
Different groups have separate stretches, so their displayed brightnesses do
not measure relative source fluxes.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    # Twelve images per PNG page; iterate every page to review all complete cutouts.
    review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
    review_directory.mkdir(parents=True, exist_ok=True)
    for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
        path = review_directory / f"all-cutouts-{page_number:03d}.png"
        sheet.save(path)
        display(sheet)
    print("Review PNGs:", review_directory)


### Changing the quality screen

The next cell changes the screen settings and creates a revision of the run.
With cached cutouts available, Gladius can reassess their pixels without
extracting them again. The new manifest records the current decisions; the
original image files keep their creation records.

A score at or above the threshold gets flagged. A larger threshold tolerates
more elongation; `None` disables the screen. The tile size and maximum shift
also affect the measurement. These settings need checking against representative
images rather than treating the example values as a scientific calibration.

Flagged cutouts remain saved. `exclude_from_stacks` controls whether they
contribute to display products; it does not remove their flags. Gladius currently
has no manual keep-or-exclude control for individual exposures.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    # Edit these values after inspecting the first audit and contact sheets.
    NEW_ACF_LIMIT = 2.0  # Demonstration only; choose based on representative images.
    NEW_TILE_SIDE = config.quality.tile_size
    NEW_LAG_RADIUS = config.quality.radius

    settings = config.model_dump(mode="json")
    settings["quality"].update(autocorrelation_limit=NEW_ACF_LIMIT,
                               tile_size=NEW_TILE_SIDE, radius=NEW_LAG_RADIUS)
    review_config = RunConfig.model_validate(settings)
    old_manifest_path = manifest_path
    manifest_path = run(review_config, progress=print)
    manifest = json.loads(manifest_path.read_text())
    print("Previous revision:", old_manifest_path.parent)
    print("Current revision:", manifest_path.parent)
    print("Work performed:", manifest["stage_counts"])
    # With the cache intact, 'extracted' is 0. Original revision files remain unchanged.
    audit = Table(rows=audit_rows(manifest_path))
    display(audit[["target", "parent", "filter", "accepted", "display_eligible", "axis_ratio", "reason"]])
    review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
    review_directory.mkdir(parents=True, exist_ok=True)
    for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
        sheet.save(review_directory / f"all-cutouts-{page_number:03d}.png")
        display(sheet)
    config = review_config


### Adding display images

A median stack combines aligned exposures. An animation shows the exposures
one at a time. A colour composite assigns three filter images to red, green,
and blue. These outputs are independent choices.

The next cells request all three kinds of display. A single-image example cannot
show noise reduction across exposures. A colour composite needs eligible images
in all three selected filters; the manifest explains any missing-filter skips.

These products are for inspection. Alignment interpolates pixels, and Gladius
does not match stellar image shapes, match photometric calibration, or propagate
uncertainties through stacking. Individual cutouts retain the original pixels
for measurements.


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    settings = config.model_dump(mode="json")
    settings["stacks"].update(enabled=True, gif_enabled=True, rgb_enabled=True,
                              blue_filter="F062", green_filter="F184", red_filter="F213")
    config = RunConfig.model_validate(settings)
    manifest_path = run(config, progress=print)
    manifest = json.loads(manifest_path.read_text())
    display_manifest = json.loads((manifest_path.parent / "stack-manifest.json").read_text())
    print("Native work:", manifest["stage_counts"])
    for category in ("products", "animations", "composites", "composite_skips", "excluded"):
        print(category, display_manifest[category])


In [ ]:
if RUN_ARCHIVE_EXAMPLE:
    from astropy.io import fits
    from IPython.display import Image as DisplayImage
    from gladius.inspection import image_preview

    # Median images: display stretch only; the FITS science values stay unchanged.
    for product in display_manifest["products"]:
        path = manifest_path.parent / product["path"]
        with fits.open(path) as hdul:
            print("Median:", product["target"], product["filter"], "inputs:", len(product["inputs"]))
            display(image_preview(hdul[0].data))
            coverage = hdul["COVERAGE"].data
            print("Contributing exposures per output pixel:", int(coverage.min()), "to", int(coverage.max()))

    # The GIF is individual aligned exposures, NOT the median shown above.
    for animation in display_manifest["animations"]:
        print("GIF:", animation["target"], animation["filter"], "frames:", len(animation["frames"]))
        display(DisplayImage(filename=str(manifest_path.parent / animation["path"])))

    # RGB has independently stretched filter channels; it is not physical colour.
    for composite in display_manifest["composites"]:
        print("RGB:", composite["target"], composite["channel_filters"])
        with Image.open(manifest_path.parent / composite["path"]) as image:
            display(image.resize((300, 300), Image.Resampling.NEAREST))


## Continuing with your own sample

The interface and Python functions use the same configuration model. A YAML file
saved from the interface can be read in Python with `load_config(path)`. Changes
in the interface do not change this notebook’s `config` variable automatically.

For archive work, `run(config, refresh_inputs=True)` refreshes the input list.
`run(config, fresh=True)` bypasses reuse and repeats the processing. Both can
require more archive access. Existing revisions keep their files.

Further details are in the [configuration guide](https://roman-gladius.readthedocs.io/en/latest/configuration.html),
[output guide](https://roman-gladius.readthedocs.io/en/latest/products.html), and
[scientific conventions](https://roman-gladius.readthedocs.io/en/latest/science.html).

This is an independent community tutorial by the Gladius contributors. The
repository’s `CITATION.cff` provides citation information. Notebook outputs can
contain restricted images and pointing information; a shared copy needs those
outputs cleared.
